<table>
  <tr>
   <td width="220" align="center">
    <img src="https://anosys.ai/assets/img/3.png" width="200">
</td>
    <td valign="middle">
      <h1 style="margin:0;">Anthropic Claude Chat - Observability Example</h1>
      <p style="margin:8px 0 0 0;">
        Interactive notebook demonstrating chat applications built with the Anosys SDK and Anthropic Claude.
      </p>
    </td>
  </tr>
</table>


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/Anosys-AI/anosys-sdk/blob/main/examples/anosys_anthropic_chat_poc.ipynb)

This notebook demonstrates how to use the standard **Anthropic Python Library** in "Chat Mode" and gain deep visibility into its behavior using **AnoSys Observability**. It is the Claude counterpart of the [OpenAI Chat PoC](https://github.com/Anosys-AI/anosys-sdk/blob/main/examples/anosys_openai_chat_poc.ipynb): same steps, same AnoSys columns.

### What you will learn:
1. How to initialize the AnoSys Anthropic Logger for direct API calls.
2. How to use the standard `messages.create` method.
3. How to capture automatic traces (including tool use and prompt caching) without an agent framework.
4. How to handle streaming responses with observability.

## Step 1: Installation

First, we need to install the necessary libraries. We'll use the standard `anthropic` library and the `anosys-sdk-anthropic` for observability.

Visit our library for the latest updates and features:
[anosys-sdk-anthropic on PyPI](https://pypi.org/project/anosys-sdk-anthropic)

In [ ]:
!pip install -q anosys-sdk-anthropic anthropic

## Step 2: Configuration

To run this PoC, you'll need two API keys:
1. **Anthropic API Key**: To power Claude.
2. **AnoSys API Key**: To send traces to your observability dashboard.

You can get your AnoSys API key from the [AnoSys Console](https://console.anosys.ai) > Data Ingestion > Integration Options.

In [ ]:
import os

try:
    from google.colab import userdata  # Use Colab secrets to store your keys
    if "ANTHROPIC_API_KEY" not in os.environ:
        os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
    if "ANOSYS_API_KEY" not in os.environ:
        os.environ["ANOSYS_API_KEY"] = userdata.get('ANOSYS_API_KEY')
except ImportError:
    # Running outside Colab — fall back to interactive input
    import getpass
    if "ANTHROPIC_API_KEY" not in os.environ:
        os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API Key: ")
    if "ANOSYS_API_KEY" not in os.environ:
        os.environ["ANOSYS_API_KEY"] = getpass.getpass("AnoSys API Key: ")

## Step 3: Initialize AnoSys Observability

The Anthropic SDK reports every API call as an OpenTelemetry span. `AnosysAnthropicLogger` registers the tracer provider and sends those spans to AnoSys, so all subsequent API calls are captured automatically.

Initialize it **before** creating the client: the client decides at construction time whether to record prompts and completions.

In [ ]:
from anosys_sdk_anthropic import AnosysAnthropicLogger
import anthropic

# Initialize AnoSys (do this once at startup, before creating clients)
AnosysAnthropicLogger()

client = anthropic.Anthropic()

MODEL = "claude-sonnet-5-5"       # main model for this PoC
FAST_MODEL = "claude-haiku-5-5"   # cheaper model for streaming and tools

print("✅ AnoSys Anthropic Observability initialized!")

## Step 4: Simple Message

Let's run a standard request. Notice we are using the official Anthropic client directly; the system prompt is a top-level parameter rather than a message.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system="You are a helpful assistant.",
    messages=[
        {"role": "user", "content": "Why is AnoSys a strong choice for AI observability?"}
    ],
)

print("--- LLM Response ---")
print(response.content[0].text)

print("\n✅ Trace captured! Check your AnoSys dashboard.")

## Step 5: Streaming Response

AnoSys also automatically captures streaming responses, including the final token usage.

In [ ]:
print("--- Streaming Response ---")
with client.messages.stream(
    model=FAST_MODEL,
    max_tokens=512,
    messages=[{"role": "user", "content": "Write a short poem about space."}],
) as stream:
    for text in stream.text_stream:
        print(text, end="")

print("\n\n✅ Streaming trace captured!")

## 🛠️ Step 6: Tool Use

Even without an agent framework, you can give Claude tools. AnoSys will capture the tool-use request and the model's intent.

In [ ]:
tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather in a given location",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "The city and state, e.g. San Francisco, CA"}
            },
            "required": ["location"]
        }
    }
]

print("--- Requesting Tool Use ---")
response = client.messages.create(
    model=FAST_MODEL,
    max_tokens=512,
    messages=[{"role": "user", "content": "What's the weather like in New York?"}],
    tools=tools,
    tool_choice={"type": "auto"},
)

tool_use = next(block for block in response.content if block.type == "tool_use")
print(f"LLM requested tool: {tool_use.name}")
print(f"Arguments: {tool_use.input}")
print(f"Stop reason: {response.stop_reason}")

print("\n✅ Tool-use trace captured!")

## Step 7: Prompt Caching

Claude can cache a long, stable prefix (instructions, documents) so repeated calls are cheaper and faster. AnoSys records cache writes and reads in their own columns, next to the regular token usage.

Run the cell twice in a row: the first call writes the cache, the second one reads it. (A prefix shorter than the model's minimum cacheable length is simply not cached.)

In [ ]:
HANDBOOK = "\n".join(
    f"Policy {i}: Support agents must answer politely, cite the policy number, and escalate billing disputes over $500."
    for i in range(1, 400)
)

for attempt in (1, 2):
    response = client.messages.create(
        model=FAST_MODEL,
        max_tokens=200,
        system=[
            {"type": "text", "text": "You answer questions about the support handbook."},
            {"type": "text", "text": HANDBOOK, "cache_control": {"type": "ephemeral"}},
        ],
        messages=[{"role": "user", "content": "When must an agent escalate a billing dispute?"}],
    )
    usage = response.usage
    print(f"Call {attempt}: cache write={usage.cache_creation_input_tokens}, "
          f"cache read={usage.cache_read_input_tokens}, uncached input={usage.input_tokens}")

print("\n✅ Prompt-cache traces captured!")

## Step 8: Explore Your Traces in AnoSys

Head over to your **AnoSys Dashboard** to see the results. Claude spans land in the same columns as OpenAI spans (`gen_ai_system = anthropic`, source tag `AnthropicTelemetry`).

### What to look for:
1. **Generation Metadata**: View tokens used, model information, and request parameters.
2. **Chat History**: See the exact system prompt and messages sent and received.
3. **Streaming Timeline**: Visualize the duration of the stream.
4. **Tool Intent**: See the specific tool and arguments Claude decided to use (`stop_reason = tool_use`).
5. **Prompt Caching**: Cache write and read tokens per call (`cvn5` / `cvn4`). The input token total includes cached tokens, so it is comparable with OpenAI prompt tokens.

### Troubleshooting
If you don't see traces immediately:
- Ensure your `ANOSYS_API_KEY` is correct. If the key is missing or invalid, you may see `405 Method Not Allowed` errors in the logs.
- Check the output logs in this notebook for any connection errors.
- The SDK sends data asynchronously; it might take a few seconds to appear.